# FSBM Chatbot — Démo Groq + LLaMA 3.3

**Notebook pédagogique pour Google Colab ou Kaggle**

Ce notebook montre comment :
1. Se connecter à Groq (API gratuite hébergeant LLaMA 3)
2. Poser des questions FSBM avec/sans RAG
3. Comparer LLaMA 3-8B vs LLaMA 3.3-70B
4. Mesurer la latence et qualité

**Pré-requis :** une clé API Groq depuis https://console.groq.com/keys (gratuit)

---

## Setup environnement

In [ ]:
# Installation des librairies nécessaires
!pip install -q groq scikit-learn

In [ ]:
# Configuration de la clé API Groq
# OPTION 1 (sur Colab) : utiliser les Secrets
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# OPTION 2 (sur Kaggle ou local) : variable d'environnement
# import os
# GROQ_API_KEY = os.environ.get('GROQ_API_KEY', 'gsk_xxx_remplace_ici')

print('Clé Groq configurée' if GROQ_API_KEY else 'AJOUTER LA CLÉ DANS LES SECRETS COLAB')

## 1. Premier appel à LLaMA 3.3-70B via Groq

In [ ]:
from groq import Groq
import time

client = Groq(api_key=GROQ_API_KEY)

start = time.perf_counter()
response = client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'Tu es un assistant FSBM. Réponds en français.'},
        {'role': 'user', 'content': 'Bonjour ! Présente-toi en 2 phrases.'}
    ],
    temperature=0.7,
    max_tokens=200,
)
latency = int((time.perf_counter() - start) * 1000)

print(f'⏱️  Latence : {latency} ms')
print(f'🪙  Tokens : {response.usage.total_tokens}')
print()
print(response.choices[0].message.content)

## 2. Démo RAG simple

On simule notre dataset FSBM (3 contextes) et on injecte le plus pertinent dans le prompt.

In [ ]:
# Mini-dataset FSBM
fsbm_contexts = {
    'inscription': '''Inscription FSBM : 1) Préinscription sur preins.univh2c.ma ; 
2) Dossier : Bac + CIN + 4 photos + reçu paiement (200 DH) ; 
3) Dépôt à la scolarité juillet-septembre ; 4) Retrait carte étudiant.
Contact : 05 22 70 46 71 / scolarite@fsbm.ac.ma''',
    'master_iads': '''Master IADS (IA & Data Science) : 2 ans, 25 places, très sélectif.
Programme : ML, Deep Learning, NLP, Vision, Big Data, MLOps.
Conditions : licence info/maths + mention AB minimum + concours + entretien.
Candidatures : mai-juillet. Coordinator : Pr. Mehdi FILALI (m.filali@fsbm.ac.ma).''',
    'bourse': '''Bourse ONOUSC : Demande sur www.onousc.ma en juillet-août.
Critères : revenus famille + distance + mention bac.
Montants : 1500-3500 DH/an selon catégorie (versement annuel).
Documents : attestation scolarité + résidence + revenus parents + CIN + RIB.''',
}

# TF-IDF simple comme retriever
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vec = TfidfVectorizer(ngram_range=(1,2))
matrix = vec.fit_transform(list(fsbm_contexts.values()))

def retrieve_top_k(query: str, k: int = 2):
    q_vec = vec.transform([query])
    sims = cosine_similarity(q_vec, matrix)[0]
    top = np.argsort(sims)[::-1][:k]
    tags = list(fsbm_contexts.keys())
    return [(tags[i], list(fsbm_contexts.values())[i], float(sims[i])) for i in top]

# Test
for ctx_tag, ctx_text, score in retrieve_top_k('Comment postuler au master IADS ?', k=2):
    print(f'📌 {ctx_tag} (score {score:.2f})')
    print(f'   {ctx_text[:120]}...')
    print()

## 3. Fonction RAG complète

In [ ]:
def chat_rag(question: str, model: str = 'llama-3.3-70b-versatile') -> str:
    # 1. Retrieval
    contexts = retrieve_top_k(question, k=2)
    
    # 2. Augmentation du prompt
    context_str = '\n\n'.join([f'--- Passage {i+1} ({tag}) ---\n{txt}' 
                                for i, (tag, txt, _) in enumerate(contexts)])
    system = f'''Tu es l'assistant officiel de la Faculté des Sciences Ben M'Sick (FSBM).
Réponds en français en utilisant UNIQUEMENT les passages ci-dessous.
Si l'info n'est pas dans les passages, dis-le franchement.

=== CONTEXTE FSBM ===
{context_str}
=== FIN CONTEXTE ==='''
    
    # 3. Generation
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': question}
        ],
        temperature=0.6,
        max_tokens=500,
    )
    return response.choices[0].message.content

# Test
print(chat_rag('Comment postuler au master IADS et y a-t-il une bourse ?'))

## 4. Comparaison LLaMA 3-8B (rapide) vs 3.3-70B (qualité)

In [ ]:
questions = [
    'Quelles sont les conditions pour le master IADS ?',
    'Salam khoya, kifash ntsajjel f FSBM ?',
    'How do I apply for a scholarship?',
]

for q in questions:
    print(f'\n{"="*60}')
    print(f'❓ {q}')
    print(f'{"="*60}')
    
    for model in ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile']:
        start = time.perf_counter()
        ans = chat_rag(q, model=model)
        latency = int((time.perf_counter() - start) * 1000)
        print(f'\n🤖 {model} ({latency}ms):')
        print(f'   {ans[:280]}...')

## 5. Conversation multi-tour avec mémoire

In [ ]:
class FSBMChatbot:
    def __init__(self, model='llama-3.3-70b-versatile'):
        self.model = model
        self.history = []  # liste de messages chat
    
    def chat(self, user_msg: str) -> str:
        contexts = retrieve_top_k(user_msg, k=2)
        ctx_str = '\n'.join([f'[{tag}] {txt[:200]}' for tag, txt, _ in contexts])
        
        messages = [
            {'role': 'system', 'content': f'''Tu es l'assistant FSBM. Garde le contexte de la conversation.
Contexte FSBM disponible :
{ctx_str}'''},
        ] + self.history[-10:] + [
            {'role': 'user', 'content': user_msg}
        ]
        
        r = client.chat.completions.create(
            model=self.model, messages=messages,
            temperature=0.6, max_tokens=400)
        
        answer = r.choices[0].message.content
        self.history.append({'role': 'user', 'content': user_msg})
        self.history.append({'role': 'assistant', 'content': answer})
        return answer

# Test conversation
bot = FSBMChatbot()
print('USER : Je suis Fatima et je suis une fille.')
print('BOT  :', bot.chat('Je suis Fatima et je suis une fille.'))
print()
print('USER : Je veux faire le master IADS.')
print('BOT  :', bot.chat('Je veux faire le master IADS.'))
print()
print('USER : Y a-t-il une bourse pour ça ?  # ambigu : ça = master IADS')
print('BOT  :', bot.chat('Y a-t-il une bourse pour ça ?'))

## Conclusion

Ce notebook a démontré :

✅ Comment connecter Groq + LLaMA 3.3 en 5 lignes  
✅ Comment implémenter un RAG simple (retrieval TF-IDF + génération LLM)  
✅ Comparaison qualité/latence entre LLaMA 3-8B et 3.3-70B  
✅ Conversation multi-tour avec mémoire et résolution d'ambiguïté  

**C'est exactement la base utilisée dans le chatbot FSBM v2** (services/chatbot-service/app/llm/).